# **Question 9: Advanced Troubleshooting & Logging**

**Focus:** **Healthchecks, Log Rotation, and Zombie Containers**

**Scenario:**
You deploy a worker container that processes background jobs from a queue. After a few days in production, the container stops processing jobs. 
* However, when you run `docker ps`, the container is still listed as **"Up"**. 
* When you attempt to see what went wrong by running `docker logs <container_id>`, your terminal completely freezes. 
* Simultaneously, an alert fires that the Docker host node's `/var` partition is at 100% disk usage.

**Question:**
1.  **The "Zombie" Container:** Why does Docker think the container is fine (Status: Up) even though the application has completely deadlocked? What specific Dockerfile instruction (or Compose equivalent) is missing that would tell Docker the app is actually broken and needs to be restarted?
2.  **The Disk Full Issue:** Why did running `docker logs` freeze your terminal, and why is the host disk full? What is Docker's default logging driver, and what daemon-level configuration must you set to prevent this from ever happening?
3.  **Production Logging Architecture:** For a senior role, relying on local log files is an anti-pattern. If you want to ship logs to a centralized system (like Datadog, Splunk, or ELK), how would you configure the Docker logging driver to do this? What happens to the container if the centralized logging server goes down and you are using a blocking logging driver?

*This question tests your battle scars with Day-2 production operations. How do you handle a system when it goes wrong?*

**Part 1: The Zombie Container**
- Why does Docker report "Up" when the application is deadlocked?
- What Docker monitors vs. what it doesn't monitor
- What Dockerfile instruction prevents this scenario?

**Part 2: The Disk Crisis**
- Why does `docker logs` freeze the terminal?
- What causes `/var` partition to fill up?
- What is Docker's default logging behavior and how do you fix it?

**Part 3: Production Logging Strategy**
- How do you ship logs to centralized systems (Datadog, Splunk, ELK)?
- What is a "blocking" logging driver?
- What happens to your app when the logging server goes down?

---

## Answer: Production Container Operations Deep Dive

### Part 1: The Zombie Container - Process vs. Application Health

#### Why Docker Shows "Up"

**What Docker Actually Monitors:**
```bash
docker ps
# Docker checks: Is PID 1 still running?
# If YES → Status: Up ✅
# If NO → Status: Exited
```

**What Docker Does NOT Monitor:**
- Application logic state
- Thread deadlocks
- Queue processing activity
- Database connection pools
- Event loop freezes
- Internal application errors

**The Core Issue:**
```
Your Worker Container:
    ├── PID 1: node server.js  ✅ Still running
    └── Application State:
            ├── Queue consumer: DEADLOCKED ❌
            ├── DB connections: FROZEN ❌
            └── Event loop: BLOCKED ❌

Docker's View: "PID 1 exists → Container is healthy" ✅
Reality: Application is completely broken ❌
```

**Key Principle:**
> Docker provides **process isolation**, not **application health monitoring**. A running process ≠ healthy application.

---

#### The Solution: HEALTHCHECK

**Dockerfile Implementation:**
```dockerfile
FROM node:18

# Add health endpoint to your app
COPY app.js /app/
WORKDIR /app

# Define healthcheck
HEALTHCHECK --interval=30s --timeout=5s --retries=3 \
  CMD curl -f http://localhost:3000/health || exit 1

CMD ["node", "server.js"]
```

**Docker Compose Implementation:**
```yaml
services:
  worker:
    image: my-worker:v1
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:3000/health"]
      interval: 30s
      timeout: 5s
      retries: 3
      start_period: 40s
```

**How It Works:**
1. **Every 30 seconds:** Docker runs the health check command
2. **Timeout 5s:** If command takes >5s, it's considered failed
3. **Retries 3:** After 3 consecutive failures → status changes to "unhealthy"
4. **Orchestrator action:** Kubernetes/Swarm can restart unhealthy containers

**Health Endpoint Example:**
```javascript
// app.js - Simple health endpoint
app.get('/health', async (req, res) => {
  try {
    // Check critical dependencies
    await db.ping();                    // DB connectivity
    await redis.ping();                 // Cache connectivity
    const queueDepth = await getQueueDepth();
    
    if (queueDepth > 10000) {          // Queue not processing
      return res.status(503).json({ status: 'unhealthy', reason: 'queue_backed_up' });
    }
    
    res.status(200).json({ status: 'healthy' });
  } catch (error) {
    res.status(503).json({ status: 'unhealthy', error: error.message });
  }
});
```

**HEALTHCHECK Lifecycle:**
```
Container starts → [starting] 
    ↓ (after start_period)
[healthy] → check passes
    ↓
[healthy] → check fails (retry 1/3)
    ↓
[healthy] → check fails (retry 2/3)
    ↓
[healthy] → check fails (retry 3/3)
    ↓
[unhealthy] → orchestrator restarts container
```

**Why This Matters:**
- ✅ Detects deadlocks (app stops responding)
- ✅ Detects stuck workers (queue depth grows)
- ✅ Detects broken dependencies (DB/Redis down)
- ✅ Enables automatic recovery
- ✅ Prevents silent failures

---

### Part 2: The Disk Crisis - Uncontrolled Log Growth

#### Why `docker logs` Froze

**Docker's Default Logging:**
```bash
# Default driver
$ docker info | grep "Logging Driver"
Logging Driver: json-file

# Where logs are stored
/var/lib/docker/containers/<container-id>/<container-id>-json.log
```

**The Problem:**
1. **No rotation by default** → logs grow infinitely
2. Your worker logs heavily → file grows to **50GB, 100GB**
3. `/var` partition fills to **100%**
4. `docker logs` tries to stream entire file → **terminal freezes**

**What Happened:**
```
Day 1: Worker starts
    ├── Logs 100 lines/sec
    └── json.log: 10MB

Day 3: Worker deadlocks
    ├── Starts error logging at 1000 lines/sec
    └── json.log: 5GB

Day 7: Disk full
    ├── json.log: 87GB
    ├── /var: 100% full
    ├── Docker daemon slows down
    └── docker logs <id> → tries to read 87GB → FREEZE
```

**System-Wide Impact:**
- ❌ Other containers can't write logs
- ❌ Docker daemon becomes unstable
- ❌ New containers fail to start
- ❌ Node becomes unusable

---

#### The Fix: Daemon-Level Log Rotation

**Configure Docker Daemon:**
```bash
# Create/edit: /etc/docker/daemon.json
{
  "log-driver": "json-file",
  "log-opts": {
    "max-size": "10m",      # Max 10MB per log file
    "max-file": "3"         # Keep 3 rotated files
  }
}
```

**Restart Docker:**
```bash
sudo systemctl restart docker
# OR
sudo service docker restart
```

**Result:**
```
Container Logs:
    ├── container-id-json.log      (current, up to 10MB)
    ├── container-id-json.log.1    (rotated, 10MB)
    └── container-id-json.log.2    (rotated, 10MB)
    
Max Total: 30MB per container
```

**Per-Container Override:**
```bash
docker run -d \
  --log-opt max-size=5m \
  --log-opt max-file=2 \
  my-worker:v1
```

**Why This Works:**
- ✅ Prevents disk exhaustion
- ✅ Keeps `docker logs` responsive
- ✅ Limits per-container footprint
- ✅ Automatic cleanup

---

### Part 3: Production Logging Architecture

#### The Anti-Pattern

**❌ Relying on `docker logs` in Production:**
```
Problem: Containers are ephemeral
    ├── Container crashes → logs may be lost
    ├── Node goes down → logs disappear
    └── Debugging becomes impossible
```

**✅ Solution: Centralized Logging**

---

#### Configuring Log Drivers

**Available Drivers:**
- `json-file` (default) - Local files
- `syslog` - System logger
- `journald` - systemd journal
- `fluentd` - Fluentd aggregator
- `gelf` - Graylog Extended Log Format
- `awslogs` - CloudWatch Logs
- `splunk` - Splunk HEC

**Example: Fluentd Setup**

**1. Daemon-Wide Configuration:**
```json
// /etc/docker/daemon.json
{
  "log-driver": "fluentd",
  "log-opts": {
    "fluentd-address": "localhost:24224",
    "tag": "docker.{{.Name}}"
  }
}
```

**2. Per-Container Configuration:**
```bash
docker run -d \
  --log-driver=fluentd \
  --log-opt fluentd-address=fluentd.example.com:24224 \
  --log-opt tag="worker.production" \
  my-worker:v1
```

**3. Docker Compose:**
```yaml
services:
  worker:
    image: my-worker:v1
    logging:
      driver: fluentd
      options:
        fluentd-address: "fluentd:24224"
        tag: "worker.{{.Name}}"
```

**Architecture:**
```
Docker Container
    ↓ (stdout/stderr)
Log Driver (fluentd)
    ↓ (TCP/UDP)
Fluentd Collector
    ↓
Elasticsearch/S3/DataDog
    ↓
Kibana/Splunk Dashboard
```

---

#### The Blocking Driver Problem

**Critical Issue: Blocking Logging**

**What Happens:**
```
1. Container writes log line
2. Log driver sends to centralized server
3. Server is down/slow/unreachable
4. Log driver BLOCKS waiting for response
5. Container's write() call BLOCKS
6. Application FREEZES
```

**Real Scenario:**
```javascript
// Your application
console.log('Processing job:', jobId);  // This line can FREEZE your app!

// If logging server is down:
// - write() blocks
// - Event loop blocks
// - Entire app freezes
```

**The Solution: Non-Blocking Mode**

```bash
docker run -d \
  --log-driver=fluentd \
  --log-opt fluentd-address=fluentd:24224 \
  --log-opt mode=non-blocking \
  --log-opt max-buffer-size=4m \
  my-worker:v1
```

**How Non-Blocking Works:**
```
Container writes log
    ↓
In-memory buffer (up to 4MB)
    ↓
Background thread sends to server
    ↓
If server down:
    ├── Buffer fills to 4MB
    ├── Oldest logs are DROPPED
    └── Container CONTINUES RUNNING ✅
```

**Trade-offs:**

| Mode | Pros | Cons |
|------|------|------|
| **Blocking** | No log loss | App can freeze if logging fails |
| **Non-blocking** | App stays available | May lose logs if server down |

**Production Recommendation:**
```json
{
  "log-driver": "fluentd",
  "log-opts": {
    "fluentd-address": "fluentd:24224",
    "mode": "non-blocking",
    "max-buffer-size": "4m",
    "fluentd-async": "true",
    "fluentd-retry-wait": "1s",
    "fluentd-max-retries": "3"
  }
}
```

**Why This Matters:**
- ✅ **Availability > Complete logs** in production
- ✅ Logging infrastructure failures don't break apps
- ✅ Buffer handles temporary network issues
- ✅ Apps remain responsive

---

## Root Cause Analysis Summary

| Problem | Root Cause | Fix | Why It Works |
|---------|------------|-----|--------------|
| **Zombie container** | No healthcheck | `HEALTHCHECK` in Dockerfile | Detects app-level failures, not just process existence |
| **Disk full** | No log rotation | `max-size` + `max-file` in daemon.json | Limits per-container disk usage |
| **Frozen `docker logs`** | Massive log file | Log rotation + centralized logging | Keeps files small, readable |
| **Log loss** | Container ephemeral | Centralized logging (fluentd/awslogs) | Durable storage outside containers |
| **App freezes on log failure** | Blocking driver | `mode=non-blocking` | Protects app availability |

---

## Production Best Practices

### 1. **Always Define HEALTHCHECK**
```dockerfile
HEALTHCHECK --interval=30s --timeout=3s --retries=3 \
  CMD curl -f http://localhost/health || exit 1
```

### 2. **Configure Log Rotation Daemon-Wide**
```json
{
  "log-driver": "json-file",
  "log-opts": {
    "max-size": "10m",
    "max-file": "3"
  }
}
```

### 3. **Use Centralized Logging in Production**
```bash
# AWS CloudWatch
--log-driver=awslogs \
--log-opt awslogs-region=us-east-1 \
--log-opt awslogs-group=myapp

# Datadog
--log-driver=fluentd \
--log-opt fluentd-address=intake.logs.datadoghq.com:10516
```

### 4. **Enable Non-Blocking Mode**
```bash
--log-opt mode=non-blocking \
--log-opt max-buffer-size=4m
```

### 5. **Monitor Disk Usage**
```bash
# Set up alerts
df -h /var | awk '{print $5}' | tail -1
# Alert if > 80%
```

---

## Interview Red Flags to Avoid

❌ "Docker automatically restarts unhealthy containers"
✅ "Docker marks them as unhealthy. Orchestrators like Kubernetes/Swarm restart them based on restart policies."

❌ "Logs are stored in `/var/log`"
✅ "Docker json-file logs are in `/var/lib/docker/containers/<id>/<id>-json.log`"

❌ "Just increase disk size"
✅ "Configure log rotation at daemon level. In production, ship logs to centralized systems."

❌ "Blocking logging ensures no data loss"
✅ "Blocking logging can freeze your app if the logging backend fails. Non-blocking mode prioritizes availability."

---

## Quick Reference Card

```bash
# Healthcheck in Dockerfile
HEALTHCHECK --interval=30s CMD curl -f http://localhost/health || exit 1

# Daemon log rotation (/etc/docker/daemon.json)
{"log-opts": {"max-size": "10m", "max-file": "3"}}

# Centralized logging (non-blocking)
docker run \
  --log-driver=fluentd \
  --log-opt mode=non-blocking \
  --log-opt max-buffer-size=4m \
  my-app

# Check container health
docker inspect --format='{{.State.Health.Status}}' <container_id>

# Check log file size
du -h /var/lib/docker/containers/*/
```